In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuração visual para os gráficos
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Exemplo carregando ficheiros serializados (.pkl) que preservam os tipos de dados
try:
    X_train = joblib.load('X_train.pkl')
    X_test = joblib.load('X_test.pkl')
    y_train = joblib.load('y_train.pkl')
    y_test = joblib.load('y_test.pkl')
    print("Dados carregados com sucesso de ficheiros .pkl!")
except FileNotFoundError:
    # Fallback caso tenham guardado em .csv
    X_train = pd.read_csv('X_train.csv')
    X_test = pd.read_csv('X_test.csv')
    y_train = pd.read_csv('y_train.csv').squeeze() # .squeeze() garante que vire uma Série
    y_test = pd.read_csv('y_test.csv').squeeze()
    print("Dados carregados com sucesso de ficheiros .csv!")

# Verificação de segurança das dimensões
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

In [ ]:
def avaliar_modelo(modelo, X_train, y_train, X_test, y_test, nome_modelo):
    """
    Treina o modelo, realiza predições e calcula métricas completas
    para os conjuntos de treino e teste.
    """
    # Treino
    modelo.fit(X_train, y_train)

    # Predições
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)

    # Dicionário para armazenar métricas de treino
    metricas_train = {
        'Accuracy': accuracy_score(y_train, y_pred_train),
        'Precision (Macro)': precision_score(y_train, y_pred_train, average='macro'),
        'Recall (Macro)': recall_score(y_train, y_pred_train, average='macro'),
        'F1-Macro': f1_score(y_train, y_pred_train, average='macro')
    }

    # Dicionário para armazenar métricas de teste
    metricas_test = {
        'Accuracy': accuracy_score(y_test, y_pred_test),
        'Precision (Macro)': precision_score(y_test, y_pred_test, average='macro'),
        'Recall (Macro)': recall_score(y_test, y_pred_test, average='macro'),
        'F1-Macro': f1_score(y_test, y_pred_test, average='macro')
    }

    # Mostrar relatórios no output
    print(f"=== {nome_modelo.upper()} ===")
    print("\n[Relatório de Classificação - Teste]")
    print(classification_report(y_test, y_pred_test))

    # Plot da Matriz de Confusão (Teste)
    cm = confusion_matrix(y_test, y_pred_test)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
    plt.title(f'Matriz de Confusão - {nome_modelo} (Teste)')
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.show()

    return metricas_train, metricas_test

In [ ]:
# Instanciando com os parâmetros definidos no planeamento
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Executando a avaliação
lr_train, lr_test = avaliar_modelo(lr_model, X_train, y_train, X_test, y_test, "Regressão Logística")

In [ ]:
# Instanciando com os parâmetros definidos no planeamento
rf_model = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1)

# Executando a avaliação
rf_train, rf_test = avaliar_modelo(rf_model, X_train, y_train, X_test, y_test, "Random Forest")

In [ ]:
# Organizando as métricas num DataFrame do Pandas para comparação direta
dados_comparativos = {
    'Métrica': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Macro'],
    'LR Treino': [lr_train['Accuracy'], lr_train['Precision (Macro)'], lr_train['Recall (Macro)'], lr_train['F1-Macro']],
    'LR Teste': [lr_test['Accuracy'], lr_test['Precision (Macro)'], lr_test['Recall (Macro)'], lr_test['F1-Macro']],
    'RF Treino': [rf_train['Accuracy'], rf_train['Precision (Macro)'], rf_train['Recall (Macro)'], rf_train['F1-Macro']],
    'RF Teste': [rf_test['Accuracy'], rf_test['Precision (Macro)'], rf_test['Recall (Macro)'], rf_test['F1-Macro']]
}

df_comparativo = pd.DataFrame(dados_comparativos)
# Arredondando para 4 casas decimais para exibição limpa
df_comparativo = df_comparativo.round(4)
df_comparativo

In [ ]:
### Discussão dos Resultados e Diagnóstico de Overfitting

1. **Análise de Overfitting (Gap Treino vs. Teste):**
   * *Regressão Logística:* Avaliar se as métricas de treino e teste estão próximas. Por ser um modelo linear, tende a sofrer menos overfitting, mas pode sofrer de underfitting se as relações forem estritamente não-lineares.
   * *Random Forest:* Verificar o gap. Se o Random Forest apresentar `F1-Macro` próximo de 1.0 no treino e um valor significativamente menor no teste, o modelo decorou os dados (overfitting severo).

2. **Desempenho nas Classes Minoritárias (Severidade 2 e 3):**
   * Como o dataset original continha um forte desbalanceamento (64.33% na classe majoritária), o uso do `class_weight='balanced'` forçou os modelos a prestarem atenção nas classes de maior severidade.
   * Analisando as Matrizes de Confusão, podemos observar se o modelo consegue identificar corretamente os acidentes graves ou se ainda gera muitos falsos positivos/negativos nessas categorias.

3. **Limitações Identificadas:**
   * A exclusão de variáveis que geravam vazamento de dados (`INJURIES`, `FATALITIES`), embora metodologicamente obrigatória, remove preditores altamente correlacionados com o target, desafiando os modelos a encontrar padrões puramente circunstanciais (clima, estrada, iluminação).
   * Possível necessidade de tunagem de hiperparâmetros (ex: limitar `max_depth` no Random Forest) caso o overfitting tenha sido impeditivo.